VERİLEN SES KAYITLARININ MFCC ÖZELLİKLERİNİN ÇIKARILMASI 

In [1]:
import librosa
import numpy as np
import os

# Ses dosyalarınızı kaydettiğiniz klasör
ses_dosyasi_klasoru = "ses_verileri"

def ozellik_cikarma(file_path):
    # Ses dosyasını yükleme
    y, sr = librosa.load(file_path, duration=5)  # 5 saniye sınırlaması
    # MFCC özelliklerini çıkarma
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    # Ortalama alarak sabit boyutlu bir vektör elde etme
    return np.mean(mfcc, axis=1)

# Tüm dosyalar için özellik çıkarma
veri = []
etiketler = []

for file in os.listdir(ses_dosyasi_klasoru):
    if file.endswith('.wav'):
        file_path = os.path.join(ses_dosyasi_klasoru, file)
        features = ozellik_cikarma(file_path)
        veri.append(features)
        etiketler.append(file.split('.')[0])  # Dosya adını etiket olarak kullanma

veri = np.array(veri)
etiketler = np.array(etiketler)


KAYDI VERİLEN KİSİNİN KİM OLDUGUNU BULMA

In [2]:
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Input
import os

# Özellik çıkarma fonksiyonu
def ozellik_cikarma(file_path):
    y, sr = librosa.load(file_path, duration=5)  # Ses dosyasını yükleme
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)  # MFCC çıkarma
    return np.mean(mfcc, axis=1)  # Ortalama ile sabit boyutlu vektör

# Veriyi yükleme
def veri_yukle(ses_klasoru):
    veri = []
    etiketler = []
    for file in os.listdir(ses_klasoru):
        if file.endswith('.wav'):
            file_path = os.path.join(ses_klasoru, file)
            features = ozellik_cikarma(file_path)
            veri.append(features)
            etiketler.append(file.split('_')[0])  # Kişi adı dosya adından alınır
    return np.array(veri), np.array(etiketler)

# Ses dosyalarının bulunduğu klasör
ses_klasoru = "ses_verileri"
X, y = veri_yukle(ses_klasoru)

# Etiketleri sayısal forma dönüştürme
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Veriyi eğitim ve test olarak ayırma
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.25, random_state=42)

# Veri türlerini kontrol etme ve düzeltme
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

# Model oluşturma
model = Sequential([
    Input(shape=(X_train.shape[1],)),  # Giriş: Özellik sayısı
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(len(np.unique(y_train)), activation='softmax')  # Çıkış: Kişi sayısı
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Modeli eğitme
model.fit(X_train, y_train, epochs=20, batch_size=4, validation_data=(X_test, y_test), verbose=0)

# Modeli kaydetme
model.save("ses_tanima_modeli.keras")
print("Model başarıyla eğitildi ve kaydedildi.")

# Yeni ses dosyasını tahmin etme
def ses_tahmin_et(model_path, file_path, label_encoder):
    model = load_model(model_path , compile=False)
    features = ozellik_cikarma(file_path).reshape(1, -1)  # Model girişine uygun boyut
    tahmin = model.predict(features)
    tahmin_kisi = label_encoder.inverse_transform([np.argmax(tahmin)])
    return tahmin_kisi[0]

# Modelin doğruluğunu ve F1 skorunu değerlendirme
y_pred = model.predict(X_test)  # Test verisi üzerindeki tahminler
y_pred_class = np.argmax(y_pred, axis=1)  # Sınıf tahminlerini al (softmax sonucu)

# F1 skoru hesaplama
f1 = f1_score(y_test, y_pred_class, average='weighted')  # weighted: Sınıf dengesizliğini göz önünde bulundurur

# Test için bir ses dosyası tahmini
test_ses_dosyasi = "ses_verileri/selin_03.wav"  # Test edilecek dosya
tahmin = ses_tahmin_et("ses_tanima_modeli.keras", test_ses_dosyasi, le)
print(f"Tahmin edilen kişi: {tahmin}")

# Modelin doğruluğunu değerlendirme
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test seti üzerindeki doğruluk(accuracy): {accuracy:.2f}")
print(f"Test seti üzerindeki F1 skoru: {f1:.2f}")



Model başarıyla eğitildi ve kaydedildi.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
Tahmin edilen kişi: selin
Test seti üzerindeki doğruluk(accuracy): 0.75
Test seti üzerindeki F1 skoru: 0.80


KONUŞAN KİSİNİN KİM OLDUGUNU ANLIK OLARAK BULMAK

In [8]:
import sounddevice as sd
import wave

def ses_kaydet(output_file, sure=5, fs=44100):
    print("Kayıt başlıyor...")
    kayit = sd.rec(int(sure * fs), samplerate=fs, channels=1, dtype='int16')
    sd.wait()  # Kayıt tamamlanana kadar bekle
    print("Kayıt tamamlandı.")
    
    # Kaydı wav formatında kaydet
    with wave.open(output_file, 'wb') as wf:
        wf.setnchannels(1)  # Mono kanal
        wf.setsampwidth(2)  # 16-bit
        wf.setframerate(fs)
        wf.writeframes(kayit.tobytes())

# Mikrofondan kayıt yap ve tahmini gerçekleştir
kayıt_dosyasi = "mikrofon_kayit.wav"
ses_kaydet(kayıt_dosyasi)

tahmin = ses_tahmin_et("ses_tanima_modeli.keras", kayıt_dosyasi, le)
print(f"Mikrofondan kaydedilen sesin tahmini: {tahmin}")


Kayıt başlıyor...
Kayıt tamamlandı.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Mikrofondan kaydedilen sesin tahmini: rumeysa


TEXTE ÇEVİRME VE KELİME SAYISI BULMA

In [4]:
import speech_recognition as sr

# Tanıyıcıyı başlatın
recognizer = sr.Recognizer()

# Ses dosyasını tanıyın
audio_file = "mikrofon_kayit.wav"

with sr.AudioFile(audio_file) as source:
    audio = recognizer.record(source)

# Türkçe ses dosyasını metne çevirin
try:
    text = recognizer.recognize_google(audio, language="tr-TR")
    print("Metin:",text)
    
    # Metin üzerinde kelime sayısı hesaplama
    words =text.split()
    word_count = len(words)
    print(f"Kelime Sayısı: {word_count}")
    
except sr.UnknownValueError:
    print("Google Speech-to-Text, sesi anlayamadı.")
except sr.RequestError as e:
    print(f"Google Speech-to-Text servisi kullanılamıyor: {e}")


Metin: mikrofondan kaydedilen sesin tamamı kayıt dosyasına
Kelime Sayısı: 6


TEXTE ÇEVİRME - KELİME SAYISI - DUYGU ANALİZİ 

In [5]:

from transformers import pipeline
import speech_recognition as sr
import warnings
import tensorflow as tf
import logging
from transformers import pipeline

# Transformers logger'ını sadece hata seviyesine çek
logging.getLogger("transformers").setLevel(logging.ERROR)

# Modelinizi yükleyin
classifier = pipeline(
    "text-classification",
    model="bert-base-uncased"
)


# Tüm TensorFlow uyarılarını kapat
tf.get_logger().setLevel('ERROR')

# Genel Python uyarılarını kapat (isteğe bağlı)
warnings.filterwarnings("ignore")

# Tanıyıcıyı başlatın
recognizer = sr.Recognizer()

# Ses dosyasını tanıyın
audio_file = "mikrofon_kayit.wav"  # Ses dosyanızın yolu

with sr.AudioFile(audio_file) as source:
    audio = recognizer.record(source)

try:
    # Ses dosyasını metne çevir
    text = recognizer.recognize_google(audio, language="tr-TR")
    print("Metin:", text)

    # Kelime sayısı hesaplama
    words = text.split()
    word_count = len(words)
    print(f"Kelime Sayısı: {word_count}")

    # Duygu analizi modelini yükle
    duygu_analizi = pipeline("sentiment-analysis", model="savasy/bert-base-turkish-sentiment-cased")

    # Duygu analizi yap
    result = duygu_analizi(text)
    
    # Sadece duygu analizi sonucunu ve metni yazdır
    print("Duygu Analizi Sonucu:", result[0]['label'], f"Skor: {result[0]['score']:.2f}")

except sr.UnknownValueError:
    print("Google Speech-to-Text, sesi anlayamadı.")
except sr.RequestError as e:
    print(f"Google Speech-to-Text servisi kullanılamıyor: {e}")





Metin: mikrofondan kaydedilen sesin tamamı kayıt dosyasına
Kelime Sayısı: 6
Duygu Analizi Sonucu: positive Skor: 0.94


TEXTE ÇEVİRME - KELİME SAYISI - DUYGU ANALİZİ (tek tek yüz üzerinden)

In [6]:
from transformers import pipeline
import warnings
import speech_recognition as sr
import warnings
import logging
from transformers import pipeline

# Transformers logger seviyesini ayarla
logging.getLogger("transformers").setLevel(logging.ERROR)


# Uyarıları devre dışı bırak
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Tanıyıcıyı başlatın
recognizer = sr.Recognizer()

# Ses dosyasını tanıyın
audio_file = "mikrofon_kayit.wav"  # Ses dosyanızın yolu

with sr.AudioFile(audio_file) as source:
    audio = recognizer.record(source)

try:
    # Ses dosyasını metne çevir
    text = recognizer.recognize_google(audio, language="tr-TR")
    print("Metin:", text)

    # Kelime sayısı hesaplama
    words = text.split()
    word_count = len(words)
    print(f"Kelime Sayısı: {word_count}")

    # Zero-shot Classification modelini yükle
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli" , multilabel=True)

    # Potansiyel duygu etiketleri
    candidate_labels = ["mutlu", "üzgün", "öfkeli", "şaşkın", "korkmuş", "heyecanlı"]

    # Duygu analizi yap
    result = classifier(text, candidate_labels, multi_class=True)

    # Sonuçları yazdır
    print("Duygu Analizi Sonuçları:")
    for label, score in zip(result['labels'], result['scores']):
        print(f"{label}: %{score * 100:.2f}")

except sr.UnknownValueError:
    print("Google Speech-to-Text, sesi anlayamadı.")
except sr.RequestError as e:
    print(f"Google Speech-to-Text servisi kullanılamıyor: {e}")


Metin: mikrofondan kaydedilen sesin tamamı kayıt dosyasına
Kelime Sayısı: 6
Duygu Analizi Sonuçları:
üzgün: %70.70
mutlu: %47.66
öfkeli: %42.59
şaşkın: %35.07
heyecanlı: %25.74
korkmuş: %18.86


TEXTE ÇEVİRME - KELİME SAYISI - DUYGU ANALİZİ (toplamı yüz )

In [7]:
from transformers import pipeline
import warnings
import speech_recognition as sr

# Uyarıları devre dışı bırak
warnings.filterwarnings("ignore")

# Tanıyıcıyı başlatın
recognizer = sr.Recognizer()

# Ses dosyasını tanıyın
audio_file = "mikrofon_kayit.wav"  # Ses dosyanızın yolu

with sr.AudioFile(audio_file) as source:
    audio = recognizer.record(source)

try:
    # Ses dosyasını metne çevir
    text = recognizer.recognize_google(audio, language="tr-TR")
    print("Metin:", text)

    # Kelime sayısı hesaplama
    words = text.split()
    word_count = len(words)
    print(f"Kelime Sayısı: {word_count}")

    # Zero-shot Classification modelini yükle
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

    # Potansiyel duygu etiketleri
    candidate_labels = ["mutlu", "üzgün", "öfkeli"]

    # Duygu analizi yap
    result = classifier(text, candidate_labels, multi_class=True)

    # Normalize etmek için toplam skoru hesapla
    total_score = sum(result['scores'])

    # Sonuçları yazdır (normalize edilmiş yüzdelerle)
    print("Duygu Analizi Sonuçları (Toplam %100):")
    for label, score in zip(result['labels'], result['scores']):
        normalized_score = (score / total_score) * 100  # Normalize et
        print(f"{label}: %{normalized_score:.2f}")

except sr.UnknownValueError:
    print("Google Speech-to-Text, sesi anlayamadı.")
except sr.RequestError as e:
    print(f"Google Speech-to-Text servisi kullanılamıyor: {e}")


Metin: mikrofondan kaydedilen sesin tamamı kayıt dosyasına
Kelime Sayısı: 6
Duygu Analizi Sonuçları (Toplam %100):
üzgün: %43.93
mutlu: %29.61
öfkeli: %26.46
